# Step 1: Import Library

In [3]:
import csv
import re
import numpy as np
import pickle

# Step 2: Read CSV

In [4]:
def read_articles(filename):
    """
    Read articles from a CSV file
    Returns a list of dictionaries, each representing an article
    """
    articles = []
    
    with open(filename, 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file)  # read file as dictionaries using the first row as field names
        for row in reader:
            article = {
                'id': int(row['id']),
                'title': row['title'],
                'content': row['content']
            }
            articles.append(article)
    return articles

# Test the function
articles = read_articles('articles.csv')
print(f"Read {len(articles)} articles")
print(f"First article: {articles[0]['title']}")

Read 20 articles
First article: AI in Healthcare


# Step 3: Cleaning text

In [5]:
def clean_text(text):
    """
    Clean text and convert it to a list of words
    """
    # 1. Convert to lowercase
    text = text.lower()
    
    # 2. Remove punctuation (keep only letters and spaces)
    # using regex: [^a-z\s] means anything that's not a letter or space
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 3. Split text into words
    words = text.split()
    
    return words

# Test the function
sample_text = "Artificial intelligence is transforming hospitals!"
cleaned_words = clean_text(sample_text)
print(f"Original text: {sample_text}")
print(f"After cleaning: {cleaned_words}")

Original text: Artificial intelligence is transforming hospitals!
After cleaning: ['artificial', 'intelligence', 'is', 'transforming', 'hospitals']


# Step 4:  Build the global vocabulary

In [6]:

def build_vocabulary(articles):
    """
    Build a list of all unique words across all articles
    """
    vocabulary = set()  # use a set to avoid duplicates
    
    for article in articles:
        # clean the article content and get words
        words = clean_text(article['content'])
        
        # add each word to the set
        for word in words:
            vocabulary.add(word)
    
    # convert the set to a sorted list
    vocabulary = sorted(list(vocabulary))
    
    return vocabulary

# Test the function
vocabulary = build_vocabulary(articles)
print(f"Number of unique words: {len(vocabulary)}")
print(f"First 10 words: {vocabulary[:10]}")

Number of unique words: 219
First 10 words: ['a', 'about', 'accuracy', 'across', 'advanced', 'advancing', 'ai', 'allow', 'allowing', 'amounts']


# Step 5: Build vectors for each article

In [7]:
def build_vectors(articles, vocabulary):
    """
    Build a vector for each article based on the global vocabulary
    Returns a list of vectors (matrix)
    """
    vectors = []
    
    for article in articles:
        # clean the article and get its words
        words = clean_text(article['content'])
        
        # convert words list to a set for fast lookup
        words_set = set(words)
        
        # build the vector (1 if the word appears, 0 otherwise)
        vector = []
        for word in vocabulary:
            if word in words_set:
                vector.append(1)
            else:
                vector.append(0)
        
        vectors.append(vector)
    
    # convert to a numpy array
    return np.array(vectors)

# Test the function
vectors = build_vectors(articles, vocabulary)
print(f"Vectors matrix shape: {vectors.shape}")
print(f"First vector: {vectors[0][:20]}...")  # first 20 elements

Vectors matrix shape: (20, 219)
First vector: [0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0]...


# Step 6: Calculate similarity matrix

In [8]:
def calculate_similarity_matrix(vectors):
    """
    Calculate the similarity matrix between all articles
    """
    n_articles = len(vectors)
    similarity_matrix = np.zeros((n_articles, n_articles))

    for i in range(n_articles):
        for j in range(n_articles):
            if i == j:
                similarity_matrix[i][j] = 1.0  # self-similarity = 1
            else:
                # compute cosine similarity using numpy
                vector_i = vectors[i]
                vector_j = vectors[j]

                # A·B (dot product)
                dot_product = np.dot(vector_i, vector_j)

                # ||A|| * ||B||
                norm_product = np.linalg.norm(vector_i) * np.linalg.norm(vector_j)

                # avoid division by zero
                if norm_product == 0:
                    similarity = 0
                else:
                    similarity = dot_product / norm_product

                similarity_matrix[i][j] = similarity

    return similarity_matrix

# Calculate similarity matrix
similarity_matrix = calculate_similarity_matrix(vectors)
print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"First 5 values in first row: {similarity_matrix[0][:5]}")

Similarity matrix shape: (20, 20)
First 5 values in first row: [1.         0.12171612 0.12524486 0.21483446 0.06454972]


# Step 7: Save results

In [9]:
def save_similarity_matrix(similarity_matrix, filename='similarities.pkl'):
    """
    Save the similarity matrix to a pickle file
    """
    with open(filename, 'wb') as file:
        pickle.dump(similarity_matrix, file)
    print(f"Saved similarity matrix to {filename}")

# Save the matrix
save_similarity_matrix(similarity_matrix)

Saved similarity matrix to similarities.pkl


# Step 8: Find most similar articles

In [10]:
def find_most_similar(article_id, similarity_matrix, articles, n=3):
    """
    Find the top-n most similar articles to the specified article
    """
    # find the index of the article by id
    article_index = None
    for i, article in enumerate(articles):
        if article['id'] == article_id:
            article_index = i
            break

    if article_index is None:
        return f"Article with id {article_id} not found"

    # get the similarity row for the article
    similarities = similarity_matrix[article_index]

    # build a list of (index, similarity) pairs excluding the article itself
    similarities_list = []
    for i, sim in enumerate(similarities):
        if i != article_index:  # exclude the article itself
            similarities_list.append((i, sim))

    # sort descending by similarity score
    similarities_list.sort(key=lambda x: x[1], reverse=True)

    # take top n articles
    top_n = similarities_list[:n]

    # display results
    print(f"Top {n} similar articles to '{articles[article_index]['title']}':")
    for i, (idx, sim) in enumerate(top_n, 1):
        print(f"{i}. '{articles[idx]['title']}' - similarity: {sim:.4f}")

    return [(articles[idx]['title'], sim) for idx, sim in top_n]

# Test the function with the first article
find_most_similar(2, similarity_matrix, articles)

Top 3 similar articles to 'Space Exploration':
1. 'Mars Missions' - similarity: 0.2520
2. 'Internet of Things' - similarity: 0.2434
3. 'Telescope Tech' - similarity: 0.2357


[('Mars Missions', np.float64(0.25197631533948484)),
 ('Internet of Things', np.float64(0.24343224778007383)),
 ('Telescope Tech', np.float64(0.23570226039551587))]